# DS 542 Sp26 Project 3 -- Math Expression Transformer

<a href="https://colab.research.google.com/github/DL4DS/sp2026/blob/main/static_files/assignments/project3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Challenge Overview

Train an attention-based **decoder-only transformer** to evaluate math expressions involving positive integers, addition, and parentheses. The model must produce step-by-step reductions of parenthesized sub-expressions until a final integer result is reached.

**Example input/output:**
```
Input:  (((1+2)+1)+8)=
Output: (((1+2)+1)+8)=((3+1)+8)=(4+8)=12
```

Each reduction step resolves all innermost parenthesized additions simultaneously, replacing `(a+b)` with its integer value. The process repeats until a single integer remains.

### Scaling Goal

A baseline model is provided that works on **3 single-digit integers** (n=3, values 1-9). The goal is to scale this model to handle:
- **More integers**: n = 2 through 5
- **Larger integers**: 1-digit (1-9), 2-digit (10-99), and 3-digit (100-999)

Accuracy must be benchmarked across 10 specific (n, digit) combinations and reported in a table.

### Environment Initialization

In [1]:
import math
import random

import torch

In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
device

device(type='cuda')

### Tokenization

The vocabulary is character-level with 17 tokens: `<bos>`, `<eos>`, `<pad>`, plus the 14 characters `()+0123456789=`. Encoding converts a string expression to a tensor of token IDs; decoding reverses it.

In [3]:
characters = "()+0123456789="
TOKENS = ["<bos>", "<eos>", "<pad>"] + [c for c in characters]
print(TOKENS)

['<bos>', '<eos>', '<pad>', '(', ')', '+', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '=']


In [4]:
TOKEN_MAP = dict((t, i) for i, t in enumerate(TOKENS))
print(TOKEN_MAP)

{'<bos>': 0, '<eos>': 1, '<pad>': 2, '(': 3, ')': 4, '+': 5, '0': 6, '1': 7, '2': 8, '3': 9, '4': 10, '5': 11, '6': 12, '7': 13, '8': 14, '9': 15, '=': 16}


In [5]:
BOS = TOKEN_MAP["<bos>"]
EOS = TOKEN_MAP["<eos>"]
PAD = TOKEN_MAP["<pad>"]

In [6]:
def decode(token_ids):
    return "".join(TOKENS[i] for i in token_ids)

decode([0, 3, 7, 5, 6, 4, 1, 2])

'<bos>(1+0)<eos><pad>'

In [7]:
def encode(s, *, eos=True):
    if s.startswith("<bos>"):
        s = s[5:]

    output = [BOS]
    output.extend(TOKEN_MAP[c] for c in s)

    if eos:
        output.append(EOS)

    return torch.tensor(output, device=device)

decode(encode("1+2=3"))

'<bos>1+2=3<eos>'

### Problem Generation

This function `generate_instance` will generate a random expression starting from `n` random integers between `value_min` and `value_max` (inclusive) and combining them with addition in a random order.
The full expression consists of multiple rounds of reductions of the innermost parentheses replacing the parenthesized addition with its integer value.
The final value after the last equals sign is the value of the original expression before the first equals sign.

Here are some example expressions.

* `(3+4)+(9+2)=(7+11)=18`
* `(((((1+2)+3)+4)+5)+6)=((((3+3)+4)+5)+6)=(((6+4)+5)+6)=((10+5)+6)=(15+6)=21`

To be clear, each reduction step should replace all the parenthesis that only contain two numbers being added.


In [8]:
# DO NOT CHANGE

def generate_instance(n, *, value_min=1, value_max=9):
    """
    Generates random training/test examples:
      1. Samples `n` random integers in `[value_min, value_max]`
      2. Randomly pairs adjacent numbers with `+` inside parentheses
      3. Iteratively reduces innermost parentheses, recording each step
      4. Joins all steps with `=` to form the full expression

      Parameters:
      - `n` — number of integers (controls expression depth/complexity)
      - `value_min`, `value_max` — control digit count (e.g., 10-99 for 2-digit)
      - `*` is a keyword-only separator. all args after must be passed as keywords
    """
    current_numbers = [random.randint(value_min, value_max) for _ in range(n)]
    current_expressions = [[str(v) for v in current_numbers]]
    current_fresh = [True for _ in current_numbers]

    while len(current_numbers) > 1:
        next_numbers = []
        next_expressions = [[] for _ in range(len(current_expressions) + 1)]
        next_fresh = []

        i = 0
        while i < len(current_numbers):
            can_merge = (i + 1 < len(current_numbers)) and (current_fresh[i] or current_fresh[i + 1])
            if can_merge and random.random() < 0.5:
                # decided to merge
                next_numbers.append(current_numbers[i] + current_numbers[i + 1])

                next_expressions[0].append(str(next_numbers[-1]))
                for j in range(len(current_expressions)):
                    next_expressions[j + 1].append(f"({current_expressions[j][i]}+{current_expressions[j][i + 1]})")

                next_fresh.append(True)
                i += 2
            else:
                # decided not to merge
                next_numbers.append(current_numbers[i])

                next_expressions[0].append(str(next_numbers[-1]))
                for j in range(len(current_expressions)):
                    next_expressions[j + 1].append(current_expressions[j][i])

                next_fresh.append(False)
                i += 1

        if len(next_numbers) < len(current_numbers):
            current_numbers = next_numbers
            current_expressions = next_expressions
            current_fresh = next_fresh

    output = '='.join(e[0] for e in reversed(current_expressions))
    return encode(output)

decode(generate_instance(5, value_min=0, value_max=999))

'<bos>(((103+(837+339))+915)+618)=(((103+1176)+915)+618)=((1279+915)+618)=(2194+618)=2812<eos>'

In [9]:
for i in range(10):
    print(decode(generate_instance(5)))

<bos>(2+(5+(8+(9+8))))=(2+(5+(8+17)))=(2+(5+25))=(2+30)=32<eos>
<bos>((7+3)+((4+5)+5))=(10+(9+5))=(10+14)=24<eos>
<bos>(((1+4)+(7+6))+1)=((5+13)+1)=(18+1)=19<eos>
<bos>(((8+8)+(6+3))+4)=((16+9)+4)=(25+4)=29<eos>
<bos>((3+(1+(4+7)))+4)=((3+(1+11))+4)=((3+12)+4)=(15+4)=19<eos>
<bos>(1+(9+(7+(2+7))))=(1+(9+(7+9)))=(1+(9+16))=(1+25)=26<eos>
<bos>(((2+4)+1)+(2+3))=((6+1)+5)=(7+5)=12<eos>
<bos>(((6+(2+7))+6)+9)=(((6+9)+6)+9)=((15+6)+9)=(21+9)=30<eos>
<bos>(((2+8)+(9+9))+1)=((10+18)+1)=(28+1)=29<eos>
<bos>(((2+3)+5)+(5+1))=((5+5)+6)=(10+6)=16<eos>


## Implement a model that generalizes to more numbers and larger numbers


### Batching

`make_batch` generates a batch of random instances, pads to uniform length, and splits into input (`x = batch[:, :-1]`) and target (`y = batch[:, 1:]`) for next-token prediction. Note that `y` is `x` shifted by one token to the left.


---

#### Implementation Note

In this function, `*args` and `**kwargs` act as **pass-through containers** — they collect whatever arguments are passed to `make_batch` and forward them directly to `generate_instance`.

```python
def make_batch(*args, batch_size=64, **kwargs):
    seqs = [generate_instance(*args, **kwargs) for _ in range(batch_size)]
```

Breaking it down:

- **`*args`** — captures any **positional** arguments (like `n`) into a tuple. When `*args` is then used in the call to `generate_instance(*args, ...)`, it **unpacks** that tuple back into positional arguments.

- **`**kwargs`** — captures any extra **keyword** arguments (like `value_min`, `value_max`) into a dictionary. When `**kwargs` is used in the call to `generate_instance(..., **kwargs)`, it **unpacks** that dictionary back into keyword arguments.

- **`batch_size=64`** — sits between `*args` and `**kwargs`, making it a keyword-only argument. It is consumed by `make_batch` itself and is **not** forwarded to `generate_instance`.

So in practice, you can call `make_batch` exactly as you would call `generate_instance`, just adding an optional `batch_size`:

```python
# Calls generate_instance(10, value_min=1, value_max=9) exactly 32 times
make_batch(10, value_min=1, value_max=9, batch_size=32)
```

This pattern is very common in Python for writing **wrapper functions** — `make_batch` doesn't need to know the full signature of `generate_instance`; it just passes everything through.

---


In [10]:
def make_batch(*args, batch_size=64, **kwargs):
    seqs = [generate_instance(*args, **kwargs) for _ in range(batch_size)]

    # `seqs` is a list of variable length 1-D tensors so we pad each tensor to
    # the same length and then stack them into a 2-D tensor.
    batch = torch.nn.utils.rnn.pad_sequence(seqs, batch_first=True, padding_value=PAD)

    # next token targets: inputs are all but last; targets are all but first
    x = batch[:, :-1]
    y = batch[:, 1:]

    return x.to(device), y.to(device)

Let's make a batch of 2 examples with 5 integers each.

In [11]:
batch_x, batch_y = make_batch(5, batch_size=2)

You can see the output (2nd sequence) is the input (1st sequence) shifted by one token to the left.

In [17]:
print(decode(batch_x[0].tolist()))
print(decode(batch_y[0].tolist()))

<bos>((3+5)+((8+7)+9))=(8+(15+9))=(8+24)=32<eos><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>
((3+5)+((8+7)+9))=(8+(15+9))=(8+24)=32<eos><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>


This next function creates a **causal attention mask** — a matrix that prevents each position in a sequence from "seeing" future tokens during attention computation.

We'll use this later in the definition of a transformer block.

In [12]:
def causal_mask(T):
    """
    Create a causal (autoregressive) attention mask of shape (T, T).

    Each position i can attend to positions 0..i but not to any future
    position j > i. This is achieved by filling the upper triangle
    (above the main diagonal) with -inf, which becomes 0 after softmax,
    effectively blocking attention to future tokens.

    Args:
        T (int): Sequence length; the mask will be of shape (T, T).

    Returns:
        torch.Tensor: A float tensor of shape (T, T) with 0.0 on and
            below the main diagonal and -inf above it.
    """
    m = torch.full((T, T), float("-inf"), device=device)
    m = torch.triu(m, diagonal=1)  # upper triangle is masked
    return m

Here's an example of a mask for a sequence of length 4.

In [13]:
mask = causal_mask(4)
mask

tensor([[0., -inf, -inf, -inf],
        [0., 0., -inf, -inf],
        [0., 0., 0., -inf],
        [0., 0., 0., 0.]], device='cuda:0')

### Model — `MathTransformer`

Here we define a GPT-style decoder-only transformer:
- **Token embedding** + **positional embedding** (learned, absolute)
- **Causal self-attention** via `TransformerEncoder` with an upper-triangular mask
- **Padding mask** to ignore `<pad>` tokens
- **Linear head** projecting hidden states to vocabulary logits
- **Greedy autoregressive generation** in `generate()`

Baseline hyperparameters: `d_model=128, nhead=4, num_layers=4, dim_ff=256, max_len=64`.

You'll want to experiment with different values for these hyperparameters.

In [42]:
# TODO: experiment with different model hyperparameters

class MathTransformer(torch.nn.Module):
    """
    A GPT-style decoder-only transformer for sequence modeling of math expressions.
    Uses learned token and positional embeddings, stacked causal self-attention
    blocks (via TransformerEncoder with an upper-triangular mask), and a linear
    language model head projecting hidden states to vocabulary logits. Weights
    are initialized with small normal distributions (std=0.02), following GPT
    conventions.
    """
    def __init__(self, d_model=256, nhead=8, num_layers=8, dim_ff=512, max_len=100, dropout=0.2):
        """
        Initialize the MathTransformer.
        Args:
            d_model (int): Dimensionality of token and positional embeddings,
                and all hidden states throughout the model. Default: 128.
            nhead (int): Number of attention heads in each TransformerEncoderLayer.
                Must evenly divide d_model. Default: 4.
            num_layers (int): Number of stacked TransformerEncoderLayer blocks.
                Default: 4.
            dim_ff (int): Hidden dimensionality of the feed-forward sublayer
                within each TransformerEncoderLayer. Default: 256.
            max_len (int): Maximum sequence length supported by the positional
                embedding table. Default: 64.
            dropout (float): Dropout probability applied within each
                TransformerEncoderLayer. Default: 0.1.
        """
        super().__init__()
        self.d_model = d_model # dimensions of the token, position embeddings 
        self.max_len = max_len # maximum sequence length supported by the positional embedding table 

        vocab_size = len(TOKENS) 

        # token + position embeddings
        self.tok_emb = torch.nn.Embedding(vocab_size, d_model, padding_idx=PAD) # Embedding are learnable parameters, 
        self.pos_emb = torch.nn.Embedding(max_len, d_model)

        # Define a TransformerEncoderLayer with the specified parameters.
        layer = torch.nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True,
        )

        # Create a TransformerEncoder with the TransformerEncoderLayer defined
        # and the specified number of layers.
        self.blocks = torch.nn.TransformerEncoder(layer, num_layers=num_layers)

        # The head of the transformer is linear layer with d_model size
        # input and vocab_size output.
        self.lm_head = torch.nn.Linear(d_model, vocab_size)

        # initialize the weights of the token and position embeddings,
        # the linear head, and the bias of the linear head.
        torch.nn.init.normal_(self.tok_emb.weight, mean=0.0, std=0.02)
        torch.nn.init.normal_(self.pos_emb.weight, mean=0.0, std=0.02)
        torch.nn.init.normal_(self.lm_head.weight, mean=0.0, std=0.02)
        torch.nn.init.zeros_(self.lm_head.bias)

    def forward(self, x):
        """
        Run a forward pass through the MathTransformer.
        Combines learned token and positional embeddings (with token embeddings
        scaled by sqrt(d_model)), then passes the result through stacked causal
        self-attention blocks using both a causal mask (to prevent attending to
        future positions) and a key padding mask (to ignore PAD tokens). The
        final hidden states are projected to vocabulary logits by the language
        model head.
        Args:
            x (torch.Tensor): Integer token index tensor of shape (N, T), where
                N is the batch size and T is the sequence length.
        Returns:
            torch.Tensor: Logits of shape (N, T, vocab_size), where each position
                contains unnormalized scores over the vocabulary. The logit at
                position i represents the model's prediction for the token
                following position i.
        """
        # x: (N, T)
        N, T = x.shape

        # Create position indices [0, 1, 2, ..., T-1] as a (1, T) tensor.
        # The .unsqueeze(0) adds the batch dimension so it can broadcast across
        #  all N sequences.
        pos = torch.arange(T, device=x.device).unsqueeze(0)  # (1, T)

        # Token embeddings scaled by sqrt(d_model) and added to positional
        # embeddings. This combines the token and positional information to
        # form the input to the transformer encoder.
        # h shape: (N, T, d_model)
        h = self.tok_emb(x) * math.sqrt(self.d_model) + self.pos_emb(pos)

        # key padding mask: -inf where PAD, 0.0 elsewhere (float, additive — same type as attn_mask)
        key_padding_mask = torch.zeros(N, T, device=x.device)
        key_padding_mask = key_padding_mask.masked_fill(x == PAD, float('-inf'))

        # causal mask for self-attention (float, -inf above diagonal)
        attn_mask = causal_mask(T) # (T, T)

        # Pass the input through the transformer encoder.
        # The encoder applies self-attention with the causal mask and ignores
        # PAD tokens.
        h = self.blocks(
            h,
            mask=attn_mask,                         # causal
            src_key_padding_mask=key_padding_mask   # pad masking
        )

        # Project the hidden states to vocabulary logits.
        logits = self.lm_head(h)  # (N, T, vocab)
        return logits

    @torch.no_grad()
    def generate(self, prefix_ids, max_new_tokens=64):
        """
        Autoregressively generate tokens following a given prefix.
        Starting from the provided prefix, repeatedly runs a forward pass,
        takes the logit at the last position, and greedily appends the
        highest-scoring token. Stops early if all sequences in the batch
        produce an EOS token, or if the sequence length reaches max_len.

        Args:
            prefix_ids (torch.Tensor): Integer token index tensor of shape
                (N, T0), where N is the batch size and T0 is the prefix length.
            max_new_tokens (int): Maximum number of new tokens to generate
                beyond the prefix. Default: 64.
        Returns:
            torch.Tensor: Integer token index tensor of shape (N, T0 + K),
                where K <= max_new_tokens is the number of tokens actually
                generated before hitting the EOS or max_len stopping condition.
        """
        self.eval()
        x = prefix_ids.clone().to(next(self.parameters()).device)  # (N, T0)
        for _ in range(max_new_tokens):
            if x.size(1) >= self.max_len:
                break
            logits = self.forward(x)[:, -1, :]   # (N, V)
            next_id = torch.argmax(logits, dim=-1, keepdim=True)  # greedy
            x = torch.cat([x, next_id], dim=1)
            if (next_id == EOS).all():
                break
        return x

# Instantiate the model as a test of the init function
test_model = MathTransformer(d_model=8, nhead=2, num_layers=2, dim_ff=128, max_len=64, dropout=0.1)

**Why are we using `TransformerEncoder` instead of `TransformerDecoder`?**

<details>
<summary>
The short answer: <b>the causal mask turns the encoder into a decoder.</b> Expand for details.
</summary>

In PyTorch's terminology, `TransformerDecoder` is specifically designed for **cross-attention** between an encoder and a decoder (the classic encoder-decoder architecture used in, e.g., translation tasks). It has two attention sublayers per block:
1. Self-attention over the decoder's own tokens
2. Cross-attention over the encoder's output

But for a **decoder-only / GPT-style** language model — which has no separate encoder to attend to — you don't need cross-attention at all. You just need self-attention with a causal mask. `TransformerEncoder` provides exactly that: stacked self-attention blocks with no cross-attention.

The key insight is:

> **`TransformerEncoder` + causal mask = decoder-only transformer**

The `mask` argument to `TransformerEncoder.forward()` is what enforces the autoregressive constraint, making it behave causally. Without the mask, it would be a bidirectional encoder (like BERT). With the causal mask, it becomes a unidirectional, left-to-right model (like GPT).

So the naming is a bit unfortunate/confusing in PyTorch:

| PyTorch class | Use case |
|---|---|
| `TransformerEncoder` | Bidirectional encoder (BERT) **or** decoder-only LM (GPT) with causal mask |
| `TransformerDecoder` | Cross-attending decoder in encoder-decoder models (T5, original Transformer) |

This model is following the GPT paradigm — a decoder-only architecture — but implements it using `TransformerEncoder` because that's the appropriate PyTorch building block when there's no encoder to cross-attend to.

</details>

> Note that unlike feed-forward fully connected and convolutional neural networks that only need an `init` and `forward` method, we also implement `generate()` for autoregressive generation.


### Initialize Model, Criterion and Optimizer

Initialize the model, criterion, and optimizer.

In [43]:
# TODO: experiment with different model, criterion, and optimizer hyperparameters
import torch.nn.functional as F

model = MathTransformer().to(device)
criterion = torch.nn.CrossEntropyLoss(ignore_index=PAD)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4)

And now we can run the training loop.

You'll probably want more diversity in the types of batches that you create.

In [44]:
XX, YY = make_batch(4, batch_size=2)
print(XX, len(XX[0]), len(XX[1]))

tensor([[ 0,  3,  3,  3,  7,  5, 14,  4,  5, 13,  4,  5, 15,  4, 16,  3,  3, 15,
          5, 13,  4,  5, 15,  4, 16,  3,  7, 12,  5, 15,  4, 16,  8, 11],
        [ 0,  3,  3, 14,  5, 10,  4,  5,  3, 12,  5, 12,  4,  4, 16,  3,  7,  8,
          5,  7,  8,  4, 16,  8, 10,  1,  2,  2,  2,  2,  2,  2,  2,  2]],
       device='cuda:0') 34 34


In [45]:
x_1, y_1 = make_batch(2, batch_size=128)
print(x_1.size())

torch.Size([128, 9])


In [46]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def make_complicated_dataset(
    batch_size: int = 128,
    lengths=(1, 2, 3, 4, 5)
):
    xs, ys = [], []

    for T in lengths:
        x, y = make_batch(T, batch_size=batch_size)
        xs.append(x.to(device=device))
        ys.append(y.to(device=device))

        # "difficult" medium range
        x_d, y_d = make_batch(T, value_min=10, value_max=99, batch_size=batch_size)
        xs.append(x_d.to(device=device))
        ys.append(y_d.to(device=device))

        # add high-range only for longer sequences (optional logic like yours)
        if T <= 3:
            x_h, y_h = make_batch(T, value_min=100, value_max=999, batch_size=batch_size)
            xs.append(x_h.to(device=device))
            ys.append(y_h.to(device=device))

    max_len = max(x.size(1) for x in xs)

    xs_padded = []
    ys_padded = []

    for x in xs:
        pad = max_len - x.size(1)
        xs_padded.append(F.pad(x, (0, pad), value=PAD))

    for y in ys:
        pad = max_len - y.size(1)
        ys_padded.append(F.pad(y, (0, pad), value=PAD))

    return xs_padded, ys_padded

In [47]:
model.train()  # set to train mode

steps_per_data_modality = 1400   # TODO: experiment with different number of steps
try: 
    for step in range(1, steps_per_data_modality+1):
        # TODO: Experiment with different batch configurations
        current_step_loss = 0.0 
        xs, ys = make_complicated_dataset(batch_size=256, lengths=[2, 3, 4, 5])
        for idx, (x, y) in enumerate(zip(xs, ys)): 
            logits = model(x) 
            loss = criterion(logits.reshape(-1, len(TOKENS)), y.reshape(-1))
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            current_step_loss += loss.item()
        if step % 20 == 0:
            print(f"step {step:4d} | loss {loss.item():.4f}")
except Exception as e: 
    pass 

step   20 | loss 1.8357
step   40 | loss 1.6885
step   60 | loss 1.5319
step   80 | loss 1.4268
step  100 | loss 1.3300
step  120 | loss 1.3117
step  140 | loss 1.2649
step  160 | loss 1.1226
step  180 | loss 1.1407
step  200 | loss 1.0019
step  220 | loss 0.9348
step  240 | loss 0.8911
step  260 | loss 0.8125
step  280 | loss 0.6385
step  300 | loss 0.6362
step  320 | loss 0.8273
step  340 | loss 0.6679
step  360 | loss 0.5707
step  380 | loss 0.5294
step  400 | loss 0.5348
step  420 | loss 0.5128
step  440 | loss 0.5145
step  460 | loss 0.5279
step  480 | loss 0.4952
step  500 | loss 0.5243
step  520 | loss 1.3465
step  540 | loss 0.6041
step  560 | loss 0.5080
step  580 | loss 0.4956
step  600 | loss 0.4969
step  620 | loss 0.4937
step  640 | loss 0.5024
step  660 | loss 0.4980
step  680 | loss 0.5004
step  700 | loss 0.4940
step  720 | loss 0.4985
step  740 | loss 1.1922
step  760 | loss 0.5110
step  780 | loss 0.4981
step  800 | loss 0.4987
step  820 | loss 0.4880
step  840 | loss

In [37]:
print(device)
model = MathTransformer() 
model.load_state_dict(torch.load("math.pt", map_location=device))
model.to(device)

steps_per_data_modality = 220   # TODO: experiment with different number of steps
try:
    for step in range(1, steps_per_data_modality+1):
        # TODO: Experiment with different batch configurations
        current_step_loss = 0.0 
        xs, ys = make_complicated_dataset(batch_size=256, lengths=[2, 3, 4, 5])
        for idx, (x, y) in enumerate(zip(xs, ys)): 
            logits = model(x) 
            loss = criterion(logits.reshape(-1, len(TOKENS)), y.reshape(-1))
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            current_step_loss += loss.item()
        if step % 20 == 0:
            print(f"step {step:4d} | loss {loss.item():.4f}")
except Exception as e: 
    print(f"Error: {e}") 

cuda
step   20 | loss 0.4916
step   40 | loss 0.4948
step   60 | loss 0.4940
step   80 | loss 0.4886
step  100 | loss 0.4920
step  120 | loss 0.4903
step  140 | loss 0.4849
step  160 | loss 0.4938
step  180 | loss 0.4884
step  200 | loss 0.4949
step  220 | loss 0.4930


In [48]:
def prepare_prompt(s):
    """
    Convert a string prompt into a token ID tensor ready for generation.
    Encodes the string into token IDs. If the string contains an '=',
    the token IDs are truncated to include everything up to and including
    the '=' token, so the model can be asked to generate the answer.

    Args:
        s (str): The input string to encode, e.g. "12+34=" or "7*8=".
    Returns:
        torch.Tensor: A long tensor of shape (1, T) containing the token
            IDs, placed on the appropriate device, ready to be passed to
            model.generate().
    """
    token_ids = encode(s)
    if '=' in s:
        token_ids = token_ids[:s.index('=')+2]
        assert token_ids[-1] == TOKEN_MAP['=']

    return torch.tensor([token_ids], dtype=torch.long, device=device)

In [49]:
def test_example(*args, verbose=True, **kwargs):
    """
    Generate a target sequence, extract the prompt (everything up to and
    including '='), run the model autoregressively from that prompt, and
    check whether the model's output matches the target.

    All positional and keyword arguments (except *verbose*) are forwarded
    to ``generate_instance`` so that callers can control the properties of
    the generated example (e.g. ``n=3`` for a 3-operand expression).

    Parameters
    ----------
    *args :
        Positional arguments passed through to ``generate_instance``.
    verbose : bool, optional
        If ``True`` (default), always print the prompt, target, actual
        output, and correctness verdict.  When ``False``, output is still
        printed whenever the model's prediction is incorrect.
    **kwargs :
        Keyword arguments passed through to ``generate_instance``.

    Returns
    -------
    bool
        ``True`` if the model's decoded output exactly matches the target
        string, ``False`` otherwise.
    """
    model.eval()

    target_token_ids = generate_instance(*args, **kwargs)
    target = decode(target_token_ids)

    prompt = target[:target.index('=')+1]
    prompt_token_ids = encode(prompt, eos=False)
    prompt_batch = prompt_token_ids.reshape(shape=(1,-1))

    actual_token_ids = model.generate(prompt_batch, max_new_tokens=46)[0]
    actual = decode(actual_token_ids)

    correct = actual == target

    if verbose or not correct:
        print("PROMPT", decode(prompt_token_ids), "\nTARGET", target, "\nACTUAL", actual, "\nCORRECT", correct)

    return correct

test_example(n=5)

PROMPT <bos>(((4+5)+(2+8))+3)= 
TARGET <bos>(((4+5)+(2+8))+3)=((9+10)+3)=(19+3)=22<eos> 
ACTUAL <bos>(((4+5)+(2+8))+3)=((9+10)+3)=(19+3)=22<eos> 
CORRECT True


True

Let's try our model with 3 numbers, which was what the original model trained on.

In [50]:
for _ in range(5):
    test_example(n=2, verbose=True, value_min=10, value_max=99)
    print("")

PROMPT <bos>(88+42)= 
TARGET <bos>(88+42)=130<eos> 
ACTUAL <bos>(88+42)=130<eos> 
CORRECT True

PROMPT <bos>(80+71)= 
TARGET <bos>(80+71)=151<eos> 
ACTUAL <bos>(80+71)=151<eos> 
CORRECT True

PROMPT <bos>(83+48)= 
TARGET <bos>(83+48)=131<eos> 
ACTUAL <bos>(83+48)=131<eos> 
CORRECT True

PROMPT <bos>(63+74)= 
TARGET <bos>(63+74)=137<eos> 
ACTUAL <bos>(63+74)=137<eos> 
CORRECT True

PROMPT <bos>(67+33)= 
TARGET <bos>(67+33)=100<eos> 
ACTUAL <bos>(67+33)=100<eos> 
CORRECT True



Let's try with 4 numbers.

The initial training configuration didn't include 4 numbers so this is _out of distribution_ of the original training data.

In [51]:
for _ in range(5):
    test_example(n=4, verbose=True)
    print("")

PROMPT <bos>((1+(8+5))+9)= 
TARGET <bos>((1+(8+5))+9)=((1+13)+9)=(14+9)=23<eos> 
ACTUAL <bos>((1+(8+5))+9)=((1+13)+9)=(14+9)=23<eos> 
CORRECT True

PROMPT <bos>((8+(5+2))+2)= 
TARGET <bos>((8+(5+2))+2)=((8+7)+2)=(15+2)=17<eos> 
ACTUAL <bos>((8+(5+2))+2)=((8+7)+2)=(15+2)=17<eos> 
CORRECT True

PROMPT <bos>(8+(2+(4+4)))= 
TARGET <bos>(8+(2+(4+4)))=(8+(2+8))=(8+10)=18<eos> 
ACTUAL <bos>(8+(2+(4+4)))=(8+(2+8))=(8+10)=18<eos> 
CORRECT True

PROMPT <bos>((4+7)+(5+8))= 
TARGET <bos>((4+7)+(5+8))=(11+13)=24<eos> 
ACTUAL <bos>((4+7)+(5+8))=(11+13)=24<eos> 
CORRECT True

PROMPT <bos>((1+8)+(3+4))= 
TARGET <bos>((1+8)+(3+4))=(9+7)=16<eos> 
ACTUAL <bos>((1+8)+(3+4))=(9+7)=16<eos> 
CORRECT True



### Benchmark your model

Test your code with different numbers of integers and numbers of input digits.
The `generate_instance` function provided uses the parameter `n` to control the number of integers, and `value_min` and `value_max` to control the range of integers.
For example, 2 input digits would correspond to `value_min=10` and `value_max=99`.

Test the accuracy on the combinations specified in the table below, and fill in your accuracy numbers in that table.
Make sure that you run enough samples for statistical significance (usually at least 1000 recommended) as your benchmarking accuracy will be checked for consistency with tests by the auto-grader.

In [52]:
# YOUR CODE HERE


## Testing two operand cases 

res2_1dig_results = []
for _ in range(1000): 
    res2_1dig = test_example(n=2, value_min=1, value_max=9, verbose=False)
    res2_1dig_results.append(res2_1dig)

res2_2dig_results = []
for _ in range(1000): 
    res2_2dig = test_example(n=2, value_min=10, value_max=99, verbose=False)
    res2_2dig_results.append(res2_2dig)
    
res2_3dig_results = []
for _ in range(1000): 
    res2_3dig = test_example(n=2, value_min=100, value_max=999, verbose=False)
    res2_3dig_results.append(res2_3dig)
                             
res3_1dig_results = []
## Testing three operand cases 
for _ in range(1000): 
    res3_1dig = test_example(n=3, value_min=1, value_max=9, verbose=False)
    res3_1dig_results.append(res3_1dig)
    
res3_2dig_results = []
for _ in range(1000): 
    res3_2dig = test_example(n=3, value_min=10, value_max=99, verbose=False)
    res3_2dig_results.append(res3_2dig)

res3_3dig_results = []
for _ in range(1000): 
    res3_3dig = test_example(n=3, value_min=100, value_max=999, verbose=False)
    res3_3dig_results.append(res3_3dig)

res4_1dig_results = []
for _ in range(1000): 
    res4_1dig = test_example(n=4, value_min=1, value_max=9, verbose=False)
    res4_1dig_results.append(res4_1dig)

res4_2dig_results = []
for _ in range(1000): 
    res4_2dig = test_example(n=4, value_min=10, value_max=99, verbose=False)
    res4_2dig_results.append(res4_2dig)

res5_1dig_results = []
for _ in range(1000): 
    res5_1dig = test_example(n=5, value_min=1, value_max=9, verbose=False)
    res5_1dig_results.append(res5_1dig)
    
res5_2dig_results = []
for _ in range(1000): 
    res5_2dig = test_example(n=5, value_min=10, value_max=99, verbose=False)
    res5_2dig_results.append(res5_2dig)

PROMPT <bos>((101+843)+657)= 
TARGET <bos>((101+843)+657)=(944+657)=1601<eos> 
ACTUAL <bos>((101+843)+657)=(944+657)=1501<eos> 
CORRECT False


In [53]:
res2_3dig_results = []
for _ in range(1000): 
    res2_3dig = test_example(n=2, value_min=100, value_max=999, verbose=False)
    res2_3dig_results.append(res2_3dig)
    
res3_3dig_results = []
for _ in range(1000): 
    res3_3dig = test_example(n=3, value_min=100, value_max=999, verbose=False)
    res3_3dig_results.append(res3_3dig)

res5_2dig_results = []
for _ in range(1000): 
    res5_2dig = test_example(n=5, value_min=10, value_max=99, verbose=False)
    res5_2dig_results.append(res5_2dig)

PROMPT <bos>((311+433)+557)= 
TARGET <bos>((311+433)+557)=(744+557)=1301<eos> 
ACTUAL <bos>((311+433)+557)=(744+557)=1201<eos> 
CORRECT False
PROMPT <bos>(((62+68)+78)+(97+97))= 
TARGET <bos>(((62+68)+78)+(97+97))=((130+78)+194)=(208+194)=402<eos> 
ACTUAL <bos>(((62+68)+78)+(97+97))=((130+78)+194)=(208+194)=302<eos> 
CORRECT False


In [55]:
print(sum(res2_1dig_results))
print(sum(res2_2dig_results))
print(sum(res2_3dig_results))

print(sum(res3_1dig_results))
print(sum(res3_2dig_results))
print(sum(res3_3dig_results))

print(sum(res4_1dig_results))
print(sum(res4_2dig_results))

print(sum(res5_1dig_results))
print(sum(res5_2dig_results))

1000
1000
1000
1000
1000
999
1000
1000
1000
999


Fill in this table.

| n | input digits | accuracy |
|---|---|-----|
| 2 | 1 | 1000 |
| 2 | 2 | 1000 |
| 2 | 3 | 1000 |
| 3 | 1 | 1000 |
| 3 | 2 | 1000 |
| 3 | 3 | 999 |
| 4 | 1 | 1000 |
| 4 | 2 | 1000 |
| 5 | 1 | 1000 |
| 5 | 2 | 999 |

Do not change the table header as the auto-grader will use it to check your results.


## Save model and implement a command line interface.

Your model will be tested automatically with a suite of examples with different numbers of values and digits matching your previous benchmark task.
For this testing, you must save your model weights and write a program to run your model.

### Save your model weights.

Save your model weights as `math.pt` to be submitted in Gradescope.

In [58]:
torch.save(model.state_dict(), "math.pt")

This saves only the learned parameters (not the model class definition), keeping the file small.

### Write a program to run your model.

Write a Python script `predict.py` that takes a single filename as input, reads each line as a prompt, generates the completion, and writes out the result to standard output.
We will invoke your program with a command like `python3 predict.py INPUT.txt` and capture the standard output for grading.

The input file will not include the special tokens such as `<bos>` or `<eos>`.
Similarly, your output should not include them either.

For example, given an input file with the following contents,
```
(((1+2)+1)+8)=
```
your program should write the following output.
```
(((1+2)+1)+8)=((3+1)+8)=(4+8)=12
```
You'll need to include all supporting code: token definitions, `encode`/`decode`, `causal_mask`, and the full `MathTransformer` class, etc.

To load the model weights, you can use the following code:

```python
model = MathTransformer() # same hyperparameters as in training#
model.load_state_dict(torch.load("math.pt", map_location=device))
model.eval()
```

**Pro Tip: Test your `predict.py` script yourself to make sure it works!!**


## Final Submission

Submit your copy of this notebook with all your code, your saved model "math.pt", and your prediction script "predict.py" to Gradescope.
